# Spark Session Initialization & Data Loading

In [75]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Instacart") \
    .config("spark.driver.memory", "6g") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()

print("Spark started")

df = spark.read.parquet("instamart_dataset.parquet")

Spark started


# Dataset Overview

In [76]:
df.count(), len(df.columns)

(32434489, 15)

# Missing Value Analysis

The null values are expected in the column 'days_since_prior_order' because there are customers who are buying first time from the store.

We need to replace nulls values with 0 or 'first_time_order'.

In [ ]:
from pyspark.sql.functions import col, when, count

df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# Basic Feature Engineering

In [79]:
from pyspark.sql.functions import col, when

df = df.withColumn("order_dow", col("order_dow").cast("int"))
df = df.withColumn("order_hour_of_day", col("order_hour_of_day").cast("int"))

df = df.fillna({"days_since_prior_order": 0})

df = df.withColumn("day_name",
    when(col("order_dow") == 0, "Sunday")
    .when(col("order_dow") == 1, "Monday")
    .when(col("order_dow") == 2, "Tuesday")
    .when(col("order_dow") == 3, "Wednesday")
    .when(col("order_dow") == 4, "Thursday")
    .when(col("order_dow") == 5, "Friday")
    .when(col("order_dow") == 6, "Saturday")
)

# Product Popularity Feature Engineering

In [80]:
from pyspark.sql.functions import count
from pyspark.sql.window import Window
from pyspark.sql.functions import ntile

product_freq = df.groupBy("product_name").agg(count("*").alias("order_count"))

window_spec = Window.orderBy(product_freq["order_count"].desc())

product_bucket = product_freq.withColumn("product_popularity",ntile(3).over(window_spec))

# Aisle Popularity Feature Engineering

In [81]:
aisle_freq = df.groupBy("aisle").agg(count("*").alias("order_count"))

window_spec = Window.orderBy(aisle_freq["order_count"].desc())

aisle_bucket = aisle_freq.withColumn("aisle_popularity",ntile(3).over(window_spec))

# Customer Behavior Feature Engineering

In [82]:
from pyspark.sql.functions import avg, countDistinct

customer_df = df.groupBy("user_id").agg(countDistinct("order_id").alias("total_orders"),avg("days_since_prior_order").alias("avg_days_between_orders"))

# Customer Segmentation

In [83]:
from pyspark.sql.functions import when

customer_df = customer_df.withColumn(
    "customer_type",
    when(col("avg_days_between_orders") <= 7, "Frequent")
    .when(col("avg_days_between_orders") <= 15, "Regular")
    .otherwise("Occasional")
)

customer_df = customer_df.withColumn(
    "loyalty_flag",
    when(col("total_orders") >= 10, "High")
    .otherwise("Low")
)

# Product Reorder Behavior

In [84]:
from pyspark.sql.functions import avg

reorder_df = df.groupBy("product_name").agg(avg("reordered").alias("product_reorder_rate"))

# Basket Size Feature Engineering

In [85]:
basket_df = df.groupBy("order_id").agg(count("product_id").alias("basket_size"))

# First Product Importance

In [86]:
first_product_df = df.filter(col("add_to_cart_order") == 1).groupBy("product_name").agg(avg("reordered").alias("first_product_reorder_rate"))

# Feature Integration (Joining All Features)

In [87]:
# Product features
df = df.join(product_bucket.select("product_name", "product_popularity"),on="product_name", how="left")

# Aisle features
df = df.join(aisle_bucket.select("aisle", "aisle_popularity"),on="aisle",how="left")

# Customer features
df = df.join(customer_df,on="user_id",how="left")

# Reorder features
df = df.join(reorder_df,on="product_name",how="left")

# Basket features
df = df.join(basket_df,on="order_id",how="left")

# First product features
df = df.join(first_product_df,on="product_name",how="left")

# BUSINESS QUESTIONS 

# 1. Department Performance Analysis

## Insight:
- Produce dominates (~9.4M orders)
- Dairy & eggs also strong

## Decision:
- Increase inventory in produce
- Optimize supply chain for perishables
- Promote high-margin items in these departments

In [89]:
from pyspark.sql.functions import count

df.groupBy("department").agg(count("*").alias("total_orders")).orderBy("total_orders", ascending=False).show(10)

+---------------+------------+
|     department|total_orders|
+---------------+------------+
|        produce|     9479291|
|     dairy eggs|     5414016|
|         snacks|     2887550|
|      beverages|     2690129|
|         frozen|     2236432|
|         pantry|     1875577|
|         bakery|     1176787|
|   canned goods|     1068058|
|           deli|     1051249|
|dry goods pasta|      866627|
+---------------+------------+
only showing top 10 rows


# 2. Aisle Performance Analysis

## Insight:
- Fresh fruits & vegetables dominate
- Essentials drive traffic
## Decision:
- Place these aisles strategically (store layout)
- Bundle with complementary items (cross-sell)

In [90]:
df.groupBy("aisle").agg(count("*").alias("total_orders")).orderBy("total_orders", ascending=False).show(10)

+--------------------+------------+
|               aisle|total_orders|
+--------------------+------------+
|        fresh fruits|     3642188|
|    fresh vegetables|     3418021|
|packaged vegetabl...|     1765313|
|              yogurt|     1452343|
|     packaged cheese|      979763|
|                milk|      891015|
|water seltzer spa...|      841533|
|      chips pretzels|      722470|
|     soy lactosefree|      638253|
|               bread|      584834|
+--------------------+------------+
only showing top 10 rows


# 3. Product Demand Analysis

## Insight:
- Bananas, organic fruits dominate
- Customers prefer healthy staples
## Decision:
- Prioritize stocking these
- Use them in recommendation systems
- Bundle: banana + milk + bread

In [91]:
df.groupBy("product_name").agg(count("*").alias("order_frequency")).orderBy("order_frequency", ascending=False).show(5)

+--------------------+---------------+
|        product_name|order_frequency|
+--------------------+---------------+
|              Banana|         472565|
|Bag of Organic Ba...|         379450|
|Organic Strawberries|         264683|
|Organic Baby Spinach|         241921|
|Organic Hass Avocado|         213584|
+--------------------+---------------+
only showing top 5 rows


# 4. Weekly Order Trends

## Insight:
- Sunday highest orders
- Thursday lowest
## Decision:
- Run offers on low days (Thursday)
- Increase staffing on weekends

In [92]:
df.groupBy("day_name").agg(count("*").alias("total_orders")).orderBy("total_orders", ascending=False).show()

+---------+------------+
| day_name|total_orders|
+---------+------------+
|   Sunday|     6209666|
|   Monday|     5665856|
| Saturday|     4500304|
|  Tuesday|     4217798|
|   Friday|     4209533|
|Wednesday|     3844117|
| Thursday|     3787215|
+---------+------------+



# 5. Hourly Order Trends

## Insight:
- Peak: 10 AM – 3 PM
## Decision:
- Optimize delivery slots
- Push ads during peak hours

In [93]:
df.groupBy("order_hour_of_day").agg(count("*").alias("total_orders")).orderBy("total_orders", ascending=False).show(10)

+-----------------+------------+
|order_hour_of_day|total_orders|
+-----------------+------------+
|               10|     2764426|
|               11|     2738582|
|               14|     2691548|
|               15|     2664533|
|               13|     2663292|
|               12|     2620847|
|               16|     2537458|
|                9|     2456713|
|               17|     2089465|
|                8|     1719973|
+-----------------+------------+
only showing top 10 rows


# 6. Customer Segmentation Distribution

## Insight:
- Majority = Regular customers
- Frequent customers are smaller but valuable
## Decision:
- Loyalty programs for frequent users
- Retarget occasional user

In [95]:
df.groupBy("customer_type").agg(count("*").alias("num_users")).show()

+-------------+---------+
|customer_type|num_users|
+-------------+---------+
|     Frequent|  9514346|
|      Regular| 16766828|
|   Occasional|  6153315|
+-------------+---------+



# 7. Customer Behavior vs Basket Size

## Insight:
- Occasional users → larger baskets
## Decision:
- Push bulk offers to occasional users
- Frequent users → smaller quick orders

In [97]:
from pyspark.sql.functions import round

df.groupBy("customer_type").agg(round(avg("basket_size"),2).alias("avg_basket")).show()

+-------------+----------+
|customer_type|avg_basket|
+-------------+----------+
|     Frequent|     14.88|
|      Regular|     15.99|
|   Occasional|     16.19|
+-------------+----------+



# 8. Reorder Behavior by Department

## Insight:
- Dairy, beverages → highest reorder
## Decision:
- These are habit products
- Use subscription model / auto-reorder

In [98]:
df.groupBy("department").agg((round(avg("reordered"),2)*100).alias("reorder_rate")).orderBy("reorder_rate", ascending=False).show(10)

+------------+-----------------+
|  department|     reorder_rate|
+------------+-----------------+
|  dairy eggs|             67.0|
|   beverages|             65.0|
|     produce|             65.0|
|      bakery|             63.0|
|        deli|             61.0|
|        pets|             60.0|
|        bulk|57.99999999999999|
|      babies|57.99999999999999|
|meat seafood|56.99999999999999|
|     alcohol|56.99999999999999|
+------------+-----------------+
only showing top 10 rows


# 9. Cart Position vs Reorder Behavior

## Insight:
- First items → highest reorder rate
## Decision:
- Show recommended products early
- Optimize UI/cart ranking

In [100]:
df.groupBy("add_to_cart_order").agg((round(avg("reordered"),2)*100).alias("reorder_rate")).orderBy("add_to_cart_order").show()

+-----------------+-----------------+
|add_to_cart_order|     reorder_rate|
+-----------------+-----------------+
|                1|             68.0|
|                2|             68.0|
|                3|             66.0|
|                4|             64.0|
|                5|             62.0|
|                6|             60.0|
|                7|             59.0|
|                8|56.99999999999999|
|                9|56.00000000000001|
|               10|55.00000000000001|
|               11|             54.0|
|               12|             53.0|
|               13|             52.0|
|               14|             52.0|
|               15|             51.0|
|               16|             50.0|
|               17|             50.0|
|               18|             49.0|
|               19|             48.0|
|               20|             48.0|
+-----------------+-----------------+
only showing top 20 rows


# 10. Popularity vs Reorder Behavior

## Insight:
- Most popular ≠ most reordered always
## Decision:
### Separate:
- High demand products
- High loyalty products

In [106]:
df.groupBy("product_popularity").agg((round(avg("reordered"),2)*100).alias("reorder_rate")).show()

+------------------+------------+
|product_popularity|reorder_rate|
+------------------+------------+
|                 1|        60.0|
|                 3|        26.0|
|                 2|        40.0|
+------------------+------------+

